In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import mne
from scipy.signal import welch
from specparam import SpectralModel
from specparam.utils.download import load_example_data
from specparam.metrics.definitions import check_metrics
import pandas as pd
import warnings
from neurodsp.sim import sim_synaptic_current
from neurodsp.utils import create_times
from neurodsp.filt import filter_signal
from neurodsp.spectral import compute_spectrum

In [7]:
DATA_DIR = Path('/Users/KeanuVentura/Desktop/sesdata')

results = {}

for i in range(1, 128):
    sub_id = f"sub-{i:03d}"
    file_path = DATA_DIR / f"subject{i:03d}" / f"{sub_id}_task-flanker_eeg.vhdr"

    if not file_path.exists():
        print(f"{sub_id}: file not found, skipping")
        continue

    try:
        raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)
        
        raw.pick_types(eeg=True)
        fs = raw.info['sfreq']


        events, event_id = mne.events_from_annotations(raw)

        event_id_clean = {
            k.split('/')[-1].replace("S", "").strip(): v
            for k, v in event_id.items()
        }
        incongruent_keys = ['21', '22']
        incong_event_id = {
            k: event_id_clean[k] for k in incongruent_keys if k in event_id_clean
        }

        if len(incong_event_id) == 0:
            print(f"{sub_id}: no incongruent events, skipping")
            continue

        stim_event_values = list(incong_event_id.values())
        stim_events = events[np.isin(events[:, 2], stim_event_values)]

        if len(stim_events) == 0:
            print(f"{sub_id}: no valid trials, skipping")
            continue

        events_corrected = stim_events.copy()
        events_corrected[:, 0] += int(0.02 * fs)

        epochs = mne.Epochs(
            raw,
            events_corrected,
            event_id=incong_event_id,
            tmin=-0.5,
            tmax=0.6,
            baseline=None,
            preload=True,
            verbose=False
        )

        epochs = epochs.pick(['Pz'])

        if len(epochs) == 0:
            print(f"{sub_id}: empty epochs, skipping")
            continue

        data = epochs.get_data()[:, 0, :]
        times = epochs.times * 1000

        pre_mask = (times >= -500) & (times <= 0)

        slopes = []
        offsets = []

        for trial in data:
            pre_data = trial[pre_mask]

            freqs, psd = welch(
                pre_data,
                fs=fs,
                nperseg=len(pre_data)
            )

            mask = (freqs >= 1) & (freqs <= 40)
            freqs = freqs[mask]
            psd = psd[mask]

            fm = SpectralModel(aperiodic_mode='fixed', verbose=False)
            fm.fit(freqs, psd)

            slopes.append(fm.get_params('aperiodic', 'exponent'))
            offsets.append(fm.get_params('aperiodic', 'offset'))

        exp_pre = np.mean(slopes)
        off_pre = np.mean(offsets)

        post_mask = (times >= 300) & (times <= 600)
        all_post = data[:, post_mask].flatten()

        freqs, psd = welch(
            all_post,
            fs=fs,
            nperseg=int(1 * fs)
        )

        mask = (freqs >= 1) & (freqs <= 40)
        freqs = freqs[mask]
        psd = psd[mask]

        fm = SpectralModel(aperiodic_mode='fixed', verbose=False)
        fm.fit(freqs, psd)

        exp_post = fm.get_params('aperiodic', 'exponent')
        off_post = fm.get_params('aperiodic', 'offset')

        epochs_erp = mne.Epochs(
            raw,
            events_corrected,
            event_id=incong_event_id,
            tmin=-0.5,
            tmax=0.6,
            baseline=(-0.2, 0),
            preload=True,
            verbose=False
        )

        epochs_erp = epochs_erp.pick(['Pz'])
        data_erp = epochs_erp.get_data()[:, 0, :]

        p3b_mask = (times >= 300) & (times <= 600)
        p3b_amp = data_erp[:, p3b_mask].mean(axis=1).mean()

        results[sub_id] = {
            'exp_pre': exp_pre,
            'off_pre': off_pre,
            'exp_post': exp_post,
            'off_post': off_post,
            'p3b_amp': p3b_amp
        }

        print(f"{sub_id}: done")

    except Exception as e:
        print(f"{sub_id}: ERROR → {e}")
        continue

df = pd.DataFrame.from_dict(results, orient='index')
df.index.name = 'subject'


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: [np.str_('Stimulus/S  2'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 21'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S111'), np.str_('Stimulus/S112'), np.str_('Stimulus/S121'), np.str_('Stimulus/S122'), np.str_('Stimulus/S211'), np.str_('Stimulus/S212'), np.str_('Stimulus/S221'), np.str_('Stimulus/S222')]


/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_38062/3350465425.py:14: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)
/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_38062/3350465425.py:14: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


sub-001: done
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: [np.str_('Stimulus/S  2'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 21'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S111'), np.str_('Stimulus/S112'), np.str_('Stimulus/S121'), np.str_('Stimulus/S122'), np.str_('Stimulus/S211'), np.str_('Stimulus/S212'), np.str_('Stimulus/S221'), np.str_('Stimulus/S222')]


/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_38062/3350465425.py:14: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)
/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_38062/3350465425.py:14: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


sub-002: done
sub-003: file not found, skipping
sub-004: file not found, skipping
sub-005: file not found, skipping
sub-006: file not found, skipping
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: [np.str_('Stimulus/S  2'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 21'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S111'), np.str_('Stimulus/S112'), np.str_('Stimulus/S121'), np.str_('Stimulus/S122'), np.str_('Stimulus/S211'), np.str_('Stimulus/S212'), np.str_('Stimulus/S221'), np.str_('Stimulus/S222')]


/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_38062/3350465425.py:14: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)
/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_38062/3350465425.py:14: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


sub-007: done
sub-008: file not found, skipping
sub-009: file not found, skipping
sub-010: file not found, skipping
sub-011: file not found, skipping
sub-012: file not found, skipping
sub-013: file not found, skipping
sub-014: file not found, skipping
sub-015: file not found, skipping
sub-016: file not found, skipping
sub-017: file not found, skipping
sub-018: file not found, skipping
sub-019: file not found, skipping
sub-020: file not found, skipping
sub-021: file not found, skipping
sub-022: file not found, skipping
sub-023: file not found, skipping
sub-024: file not found, skipping
sub-025: file not found, skipping
sub-026: file not found, skipping
sub-027: file not found, skipping
sub-028: file not found, skipping
sub-029: file not found, skipping
sub-030: file not found, skipping
sub-031: file not found, skipping
sub-032: file not found, skipping
sub-033: file not found, skipping
sub-034: file not found, skipping
sub-035: file not found, skipping
sub-036: file not found, skipping


In [8]:
df.head()

,exp_pre,off_pre,exp_post,off_post,p3b_amp
subject,,,,,
sub-001,1.385262,-11.398665,2.059165,-9.036426,5.334480e-06
sub-002,1.261218,-11.766260,1.835526,-10.265721,-3.118020e-06
sub-007,1.232071,-11.810731,1.872106,-10.060469,-4.039004e-07


In [12]:
import pandas as pd

participants = pd.read_csv("participants.txt", sep='\t')

participants_clean = participants[[
    'participant_id',
    'Subjective_SES', # self-reported childhood ses
    "Highest_Edu_Grouped",
    'Highest_Adult_Edu_Grouped',
    'Age_Grouped',
    'Gender',
    'FlankerEEG', # whether they participated in flanker
    'fs1', 'fs2', 'fs3', 'fs4', 'fs5', 'fs6', # food security
    'hnc1a', 'hnc1b', 'hnc1c', 'hnc1d', # home neighbordhood characteristics
    'hnc2a', 'hnc2b',
    'hnc3', 'hnc4',
    'ADHD_binary'
]]

edu_map = {
    "High School or Less": 0,
    "High School": 1,
    "Some College or Associate Degree": 2,
    "Bachelor's Degree or Higher": 3
}

participants_clean['edu'] = participants_clean[
    "Highest_Edu_Grouped"
].map(edu_map)

participants_clean['parent_edu'] = participants_clean[
    'Highest_Adult_Edu_Grouped'
].map(edu_map)

fs_map = {
    "often true": 2,
    "sometimes true": 1,
    "never true": 0,
    "yes": 1,
    "no": 0,
    "only 1 to 2 months": 1,
    "some months but not every month": 1,
    "almost every month": 2
}

for col in ['fs1','fs2','fs3','fs4','fs5','fs6']:
    participants_clean[col] = participants_clean[col].map(fs_map)

participants_clean['food_insecurity'] = participants_clean[
    ['fs1','fs2','fs3','fs4','fs5','fs6']
].mean(axis=1)

hnc_binary_map = {
    "yes": 1,
    "no": 0
}

for col in ['hnc1a','hnc1b','hnc1c','hnc1d','hnc2a','hnc2b','hnc3']:
    participants_clean[col] = participants_clean[col].map(hnc_binary_map)

hnc4_map = {
    "very safe": 0,
    "somewhat safe": 1,
    "somewhat unsafe": 2,
    "very unsafe": 3
}

participants_clean['hnc4'] = participants_clean['hnc4'].map(hnc4_map)

participants_clean['neighborhood_score'] = participants_clean[
    ['hnc1a','hnc1b','hnc1c','hnc1d','hnc2a','hnc2b','hnc3','hnc4']
].mean(axis=1)

"""
participants_clean['gender_binary'] = participants_clean['Gender'].map({
    'male': 0,
    'female': 1
})
"""

participants_clean = participants_clean[participants_clean['FlankerEEG'] == 1]

df_clean = df.reset_index().rename(columns={'subject': 'participant_id'})

df_final = df_clean.merge(
    participants_clean,
    on='participant_id'
)

df_final['p3b_amp'] = df_final['p3b_amp'] * 1e6

df_final = df_final[[
    'participant_id',
    'p3b_amp',
    'exp_pre', 'exp_post',
    'off_pre', 'off_post',

    'Subjective_SES',
    'edu',
    'parent_edu',
    'food_insecurity',
    'neighborhood_score',

    'Age_Grouped',
    'Gender', 
    'ADHD_binary'
]]

df_final = df_final.set_index('participant_id')
df_final

,p3b_amp,exp_pre,exp_post,off_pre,off_post,Subjective_SES,edu,parent_edu,food_insecurity,neighborhood_score,Age_Grouped,Gender,ADHD_binary
participant_id,,,,,,,,,,,,,
sub-001,5.33448,1.385262,2.059165,-11.398665,-9.036426,5.0,1,2,1.166667,0.125,18 to 22,female,0.0
sub-002,-3.11802,1.261218,1.835526,-11.766260,-10.265721,3.0,3,2,0.333333,0.125,23 to 26,male,0.0
sub-007,-0.40390,1.232071,1.872106,-11.810731,-10.060469,3.0,2,2,0.400000,0.125,18 to 22,female,0.0


In [10]:
print(df.head())
print(df.index)
print(df.columns)
print(df.dtypes)

          exp_pre    off_pre  exp_post   off_post       p3b_amp
subject                                                        
sub-001  1.385262 -11.398665  2.059165  -9.036426  5.334480e-06
sub-002  1.261218 -11.766260  1.835526 -10.265721 -3.118020e-06
sub-007  1.232071 -11.810731  1.872106 -10.060469 -4.039004e-07
Index(['sub-001', 'sub-002', 'sub-007'], dtype='str', name='subject')
Index(['exp_pre', 'off_pre', 'exp_post', 'off_post', 'p3b_amp'], dtype='str')
exp_pre     float64
off_pre     float64
exp_post    float64
off_post    float64
p3b_amp     float64
dtype: object


### Variable Definitions

* **participant_id**
  Subject identifier (e.g., sub-001)

* **p3b_amp**
  Average ERP amplitude at electrode Pz from 300–600 ms post-stimulus (in microvolts); reflects attentional and cognitive processing

* **exp_pre**
  Aperiodic exponent (spectral slope) from EEG power spectrum during pre-stimulus window (−500 to 0 ms); reflects baseline neural state

* **exp_post**
  Aperiodic exponent during post-stimulus window (300–600 ms); reflects task-evoked neural state

* **off_pre**
  Aperiodic offset (broadband power level) during pre-stimulus window (−500 to 0 ms)

* **off_post**
  Aperiodic offset during post-stimulus window (300–600 ms)

* **Subjective_SES**
  Self-reported childhood socioeconomic status (ordinal scale; higher values indicate higher perceived status)

* **edu**
  Participant’s own education level (ordinal):

  * 0 = High School or Less
  * 1 = High School
  * 2 = Some College or Associate Degree
  * 3 = Bachelor’s Degree or Higher

* **parent_edu**
  Highest parental/guardian education level during childhood (ordinal proxy for objective SES):

  * 0 = High School or Less
  * 1 = High School
  * 2 = Some College or Associate Degree
  * 3 = Bachelor’s Degree or Higher

* **food_insecurity**
  Mean score across six food security items (fs1–fs6) assessing access to adequate food

  * 0 = no insecurity (e.g., “never true”, “no”)
  * 1 = moderate insecurity (e.g., “sometimes true”, occasional months)
  * 2 = high insecurity (e.g., “often true”, frequent occurrence)
    Higher values indicate greater food insecurity

* **neighborhood_score**
  Mean score across eight housing and neighborhood condition items (hnc1a–hnc4) assessing environmental quality

  * Binary items (housing/neighborhood problems): 0 = no problem, 1 = problem present
  * Safety perception: 0 = very safe → 3 = very unsafe
    Higher values indicate worse housing and neighborhood conditions

* **Age_Grouped**
  Participant age category (e.g., 18–22, 23–26)

* **Gender**
  Self-reported gender (male/female)
